# MAGAT maintainers: measure the budget levels

**Run this only on the reference machine (the R7615 for Tribes), and only deliberately.** Every other
machine, and every ordinary run, uses the published levels: `run_magat.ipynb` reads them and never
measures them.

A budget level is a number of compute units that is a round amount of thinking time on the reference
machine: 500, 1000 and 5000 ms of a whole turn, single-threaded, averaged over positions spread across
the portfolio ([units as time](../../docs/CURRENCY.md#units-as-time)). The levels are published in
`games/<game>/levels.json`, with `levels.md` and the record in `levels-record/`.

Every currency, check and experiment is measured at the published levels. New levels change what a unit
means, so **every currency must then be measured again, under a new name**. Measure again only after a
change to what a turn costs, such as a new engine commit or a new reference machine. The measurement writes
to the data root and never touches the repository; publishing new levels is a pull request.

## Parameters and helpers

In [ ]:
import os, sys, subprocess, pathlib, datetime, zipfile, glob

MAGAT_ROOT = ""     # leave empty: auto-detected. Set an absolute path only if the banner says so.

def _find_root():
    if MAGAT_ROOT:
        return pathlib.Path(MAGAT_ROOT).resolve()
    here = pathlib.Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "magat.py").is_file() and (base / "core" / "src").is_dir():
            return base
        for child in base.glob("*"):
            if child.is_dir() and (child / "magat.py").is_file() and (child / "core" / "src").is_dir():
                return child
    raise RuntimeError(
        f"Cannot find the MAGAT repository at or above {here}. A Jupyter kernel's working directory "
        "is the notebook's own folder (and it keeps the OLD folder if the file was moved without a "
        "kernel restart). Restart the kernel, or set MAGAT_ROOT at the top of this cell.")

ROOT = _find_root()
os.chdir(ROOT)

# Everything a run produces lives OUTSIDE the clone, so a merge and a re-clone never touch it.
DATA = pathlib.Path(os.environ.get("MAGAT_DATA") or (pathlib.Path.home() / "magat_runs")).resolve()
os.environ["MAGAT_DATA"] = str(DATA)
LOGS = DATA / "logs"
BUNDLES = DATA / "bundles"
for _d in (LOGS, BUNDLES):
    _d.mkdir(parents=True, exist_ok=True)
assert ROOT not in DATA.parents and DATA != ROOT, f"data root {DATA} must be outside the clone {ROOT}"

GAME = "tribes"
PY = sys.executable
PUBLISHED_PORTFOLIO = ROOT / "games" / GAME / "portfolio" / "v1"
PUBLISHED_WEIGHTS = ROOT / "games" / GAME / "calibration" / "r7615" / "compute_weights.properties"
# Workers for parallel stages. Half the logical CPUs (SMT siblings add heat, not throughput) less two
# for the machine; MAGAT_WORKERS overrides. The machine profile (milestone 3) replaces this guess.
WORKERS = int(os.environ.get("MAGAT_WORKERS") or max(1, (os.cpu_count() or 2) // 2 - 2))
# This machine's own currency: <data>/calibration/<game>-<name>. The published R7615 currency stays in
# the repository; a new calibration of the R7615 is published beside it under this name, never over it.
# A currency keeps the levels it was first measured at: "r7615-time" is the first at the published levels
# (games/<game>/levels.json); "r7615-v2" was the name for one at the thesis's 40,000 / 80,000 / 400,000.
CURRENCY_NAME = os.environ.get("MAGAT_CURRENCY") or "r7615-time"

def magat(*args, check=True):
    """Run one launcher command; the launcher tees its own output to <DATA>/logs/<name>.log."""
    cmd = [PY, str(ROOT / "magat.py"), *map(str, args)]
    print("$ " + " ".join(cmd[2:]), flush=True)
    rc = subprocess.call(cmd)
    if check and rc != 0:
        raise RuntimeError(f"exit {rc}: magat {' '.join(map(str, args))}")
    return rc

def latest_log(name):
    txt = (LOGS / f"{name}.log").read_text(encoding="utf-8", errors="replace")
    return txt.rsplit("\n>>>>> ", 1)[-1]

def assert_in_log(name, needle):
    assert needle in latest_log(name), f"'{needle}' not found in the latest {name} run - read {LOGS / (name + '.log')}"
    print(f"[gate] '{needle}' confirmed in {name}")

def summary(results, until="## Budget multiplier"):
    """Print an analysis's SUMMARY.md up to a section (the provenance, the side by side and the headline)."""
    text = (pathlib.Path(results) / "SUMMARY.md").read_text(encoding="utf-8")
    cut = text.find(until)
    print(text if cut < 0 else text[:cut] + f"... the rest in {pathlib.Path(results) / 'SUMMARY.md'}")

def bundle(name, items, exclude=("seqlog_", "mutlog_", "_running.lock")):
    """Zip the pre-specified files of a stage into <DATA>/bundles/<name>.zip."""
    out = BUNDLES / f"{name}.zip"
    if out.exists():
        out.unlink()
    files = []
    for it in items:
        p = pathlib.Path(it)
        if not p.is_absolute():
            p = DATA / p
        if p.is_dir():
            files += [q for q in p.rglob("*") if q.is_file()]
        elif p.exists():
            files.append(p)
        else:
            files += [pathlib.Path(h) for h in glob.glob(str(p)) if pathlib.Path(h).is_file()]
    files = sorted({f for f in files if not any(x in f.name for x in exclude)})
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f, str(f.relative_to(DATA)) if DATA in f.parents else f.name)
    print(f"{len(files)} files -> {out}")
    return out

print("repository :", ROOT)
print("data root  :", DATA, "(outside the clone)")
print("python     :", PY)
print("workers    :", WORKERS)

In [ ]:
magat("build")
magat("machine")

## Measure

Set `REMEASURE = True` to measure; left `False`, the cell only shows the published levels. The machine
must be idle: nothing else of yours may run while it measures.

In [ ]:
# Measure the levels again: the units that are LEVEL_TIMES milliseconds of a whole turn of the reference
# arm (its initialization included), single-threaded, averaged over LEVEL_POSITIONS positions spread across
# the portfolio, within 5%, rounded to 500 units. About an hour on the R7615. It writes
# <data>/calibration/<game>-levels and never touches the published file; publishing is a pull request.
import json
REMEASURE = False        # True only on the reference machine, deliberately: see the cell above
LEVEL_TIMES = "500,1000,5000"
LEVEL_POSITIONS = 64
PUBLISHED_LEVELS = ROOT / "games" / GAME / "levels.json"
MEASURED = DATA / "calibration" / f"{GAME}-levels"
BUILT = DATA / "portfolio" / GAME
PORTFOLIO = BUILT if (BUILT / "_split_dev.txt").exists() else PUBLISHED_PORTFOLIO
print(f"portfolio: {PORTFOLIO}" + ("" if PORTFOLIO == BUILT else "   (the published one: run_magat.ipynb builds this machine's own)"))
if PUBLISHED_LEVELS.is_file():
    print(f"published: {PUBLISHED_LEVELS}")
    print((PUBLISHED_LEVELS.parent / "levels.md").read_text(encoding="utf-8"))
if not REMEASURE:
    print("REMEASURE is False: nothing measured. The published levels stand.")
elif (MEASURED / "levels.json").exists():
    print(f"[skip] {MEASURED / 'levels.json'} exists: rename that folder to measure again")
else:
    magat("calibrate-time", "--game", GAME, "--portfolio", PORTFOLIO, "--times", LEVEL_TIMES,
          "--positions", LEVEL_POSITIONS, "--remeasure")

In [ ]:
# The measured levels beside the published ones, and the bundle to publish from.
if (MEASURED / "levels.json").exists():
    new = json.loads((MEASURED / "levels.json").read_text(encoding="utf-8"))
    print((MEASURED / "SUMMARY.md").read_text(encoding="utf-8"))
    if PUBLISHED_LEVELS.is_file():
        old = {lv["time_ms"]: lv["units"] for lv in json.loads(PUBLISHED_LEVELS.read_text(encoding="utf-8"))["levels"]}
        for lv in new["levels"]:
            was = old.get(lv["time_ms"])
            print(f"  {lv['time_ms']:>5,} ms: {lv['units']:>9,} units" + (f"   (published {was:,}, {100 * (lv['units'] / was - 1):+.1f}%)" if was else ""))
    bundle("levels", [MEASURED, LOGS / "calibrate-time.log"])
else:
    print(f"nothing measured in {MEASURED}")

## Publish

Bundle `levels` holds the measurement. To publish it, in a pull request:
1. `levels.json` becomes `games/<game>/levels.json`, `SUMMARY.md` becomes `games/<game>/levels.md`, and
   `probe.json` and `iter-*/` replace `games/<game>/levels-record/` (update its README).
2. Set the levels of every spec that runs at the published levels (`experiments/<game>/reference-ladder.json`),
   and the expected units in `PublishedLevelsTest`.
3. Every currency is then measured again, under a new name (`CURRENCY_NAME` in `run_magat.ipynb`).